<h1>PPDB Tests</h1>

<h3>Imports</h3>

In [ ]:
from abc import ABC
from pathlib import Path
import io
import requests
import time

from astropy.table import Table
from pyvo.dal import AsyncTAPJob, TAPService
from pyvo.dal.tap import TAPService
import pandas as pd
import pyvo

from lsst.rsp import RSPClient, get_tap_service, get_service_url, get_access_token

<h3>Setup</h3>

Get the PPDB TAP service.

In [ ]:
# service = get_tap_service("ppdbtap")

In [ ]:
url = get_service_url("tap", "prompt")

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {get_access_token()}"
})
service = TAPService(url, session=session)

Use this instead if running outside of the RSP (requires a valid token from the <a href="https://data-int.lsst.cloud/api-aspect">API aspect</a>).

In [ ]:
#import pyvo
#import requests
#import os

#token = os.environ["RSP_TOKEN"]
#session = requests.Session()
#session.headers["Authorization"] = f"Bearer {token}"

#rsp_tap_url = "https://data-int.lsst.cloud/api/ppdbtap"
#service = pyvo.dal.TAPService(rsp_tap_url, session=session)

Define a helper method for submitting an async job to the TAP service and getting the results.

In [ ]:
def query(sql: str):
    job = service.submit_job(sql)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    if job.phase == "ERROR":
        job.raise_if_error() 
    
    results = job.fetch_result().to_table()
    return results

<h2>Utilities</h2>

In [ ]:
class JobRunner(ABC):

    job = None
    result = None

    _start_time = None
    _end_time = None
    
    _job_elapsed = None
    _fetch_elapsed = None

    prepend = ""
    
    def __init__(self, service):
        if service is None:
            raise RuntimeError("service cannot be None")
        self.service = service
    
    def run_job(self, sql, print_timing = True):
        ...

    def fetch_result(self, job):
        ...

    def execute(self, sql):        
        self.run_job(sql)
        self.print_job_elapsed()

        runner.fetch_result()
        runner.print_fetch_elapsed()
        runner.print_result_count()

    def _time_start(self):
        self._start_time = time.time()

    def _time_end(self):
        self._end_time = time.time()

    def _time_elapsed(self):
        return self._end_time - self._start_time
    
    def print_job_elapsed(self):
        print(f"{self.prepend}Job took {self._job_elapsed:.2f} seconds")

    def print_fetch_elapsed(self):
        print(f"{self.prepend}Fetch took {self._fetch_elapsed:.2f} seconds")

    def print_result_count(self):
        print(f"{self.prepend}Retrieved {len(self.result)} rows")

class VOTableJobRunner(JobRunner):

    def run_job(self, sql):
        self._time_start()
        job = self.service.submit_job(sql)
        job.run()
        job.wait(phases=['COMPLETED', 'ERROR'])
        if job.phase == "ERROR":
            job.raise_if_error()
        job_end = time.time()
        self._time_end()
        self._job_elapsed = self._time_elapsed()
        self.job = job

    def fetch_result(self):
        self._time_start()
        self.result = self.job.fetch_result()
        self._time_end()
        self._fetch_elapsed = self._time_elapsed()

class VOParquetJobRunner(JobRunner):

    def run_job(self, sql):
        self._time_start()
        job = AsyncTAPJob.create(
            self.service.baseurl,
            sql,
            RESPONSEFORMAT="application/vnd.apache.parquet",
            session=session
        )
        job = job.run().wait()
        self._time_end()
        self._job_elapsed = self._time_elapsed()
        self.job = job

    def fetch_result(self):
        self._time_start()
        response = session.get(self.job.result_uri, stream=True)
        table = Table.read(io.BytesIO(response.content), format="parquet.votable")
        self._time_end()
        self._fetch_elapsed = self._time_elapsed()
        self.result = table

In [ ]:
def export_ppdb_table(
    table_name: str,
    export_dir: str = "ppdb_export_data",
):
    """Export PPDB table data to parquet files, one per day.

    Days will be skipped if there is a parquet file already present 
    with the correct record count.

    Parameters
    ----------
    table_name
        Name of table to export such as "DiaObject" or "DiaSource".
    export_dir
        Directory where parquet files should be written.
        Defaults to ``ppdb_export_data.
    """
    print(f"Starting export of {table_name} table...\n")
    export_start = time.time()
    
    # Create export directory (or use existing).
    Path(export_dir).mkdir(exist_ok=True)

    # Determine which timing column to use for the table.
    if table_name == "DiaObject":
        ts_col = "validityStartMjdTai"
    elif table_name == "DiaSource" or table_name == "DiaForcedSource":
        ts_col = "midpointMjdTai"
    else:
        raise Exception(f"Unsupported table: {table_name}")

    # Get a list of days (MJD TAI format) with data.
    day_results = query(
        f"""
        SELECT FLOOR({ts_col}) AS day_mjd_tai,
            COUNT(*) AS record_count
        FROM ppdb.{table_name}
        GROUP BY day_mjd_tai ORDER BY day_mjd_tai
        """)
    days = [d for d in day_results["day_mjd_tai"]]
    record_counts = [c for c in day_results["record_count"]]
    
    # Loop over the days with data and process them.
    for day, record_count in zip(days, record_counts, strict=True):

        print(f"Processing day: {day}")
        print(f"  Expected record count: {record_count}")
        
        # Make directory for this day.
        output_dir = Path(export_dir, str(int(day)))
        output_dir.mkdir(exist_ok=True)
        output_path = output_dir / f"{table_name}.parquet"
    
        # Check for and verify an existing output file and skip if exists
        # with correct record count.
        if output_path.exists():
            print(f"  Parquet file already exists: {output_path}")
            try:
                df = pd.read_parquet(output_path)
                parquet_record_count = len(df)
                print(f"  Existing parquet file has {parquet_record_count} records.")
                if parquet_record_count == record_count:
                    print("  Skipping this day - parquet file with correct record count already exists.")
                    continue
                else:
                    print(f"  Record count mismatch: {parquet_record_count} != {record_count}")
                    print("  File will be recreated.")
            except Exception as e:
                # Probably an invalid parquet file
                print(e)
        
        # Get data for the day from the TAP service.
        sql = f"SELECT * FROM ppdb.{table_name} WHERE FLOOR({ts_col}) = {day}"

        # Run the SQL job.
        job_start = time.time()
        job = service.submit_job(sql)
        job.run()
        job.wait(phases=['COMPLETED', 'ERROR'])
        if job.phase == "ERROR":
            job.raise_if_error()
        job_end = time.time()
        job_elapsed = job_end - job_start
        print(f"  Job took {job_elapsed:.2f} seconds")

        # Fetch the data.
        fetch_start = time.time()
        table_data = job.fetch_result().to_table()
        fetch_end = time.time() 
        fetch_elapsed = fetch_end - fetch_start
        print(f"  Fetch took {fetch_elapsed:.2f} seconds")
            
        # Write the entire day's data to a parquet file.
        parq_start = time.time()
        table_data.write(output_path, format="parquet", overwrite=True)
        parq_end = time.time()
        parq_elapsed = parq_end - parq_start
        print(f"  Wrote table data to '{output_path}' in {parq_elapsed:.2f} seconds")

    export_end = time.time()
    export_elapsed = export_end - export_start
    print(f"Export of {table_name} completed in {export_elapsed:.0f} seconds.")

In [ ]:
def export_ppdb_table(
    table_name: str,
    export_dir: str = "ppdb_export_data",
):
    """Export PPDB table data to parquet files, one per day.

    Days will be skipped if there is a parquet file already present 
    with the correct record count.

    Parameters
    ----------
    table_name
        Name of table to export such as "DiaObject" or "DiaSource".
    export_dir
        Directory where parquet files should be written.
        Defaults to ``ppdb_export_data.
    """
    print(f"Starting export of {table_name} table...\n")
    export_start = time.time()
    
    # Create export directory (or use existing).
    Path(export_dir).mkdir(exist_ok=True)

    # Determine which timing column to use for the table.
    if table_name == "DiaObject":
        ts_col = "validityStartMjdTai"
    elif table_name == "DiaSource" or table_name == "DiaForcedSource":
        ts_col = "midpointMjdTai"
    else:
        raise Exception(f"Unsupported table: {table_name}")

    # Get a list of days (MJD TAI format) with data.
    day_results = query(
        f"""
        SELECT FLOOR({ts_col}) AS day_mjd_tai,
            COUNT(*) AS record_count
        FROM ppdb.{table_name}
        GROUP BY day_mjd_tai ORDER BY day_mjd_tai
        """)
    days = [d for d in day_results["day_mjd_tai"]]
    record_counts = [c for c in day_results["record_count"]]
    
    # Loop over the days with data and process them.
    for day, record_count in zip(days, record_counts, strict=True):

        print(f"Processing day: {day}")
        print(f"  Expected record count: {record_count}")
        
        # Make directory for this day.
        output_dir = Path(export_dir, str(int(day)))
        output_dir.mkdir(exist_ok=True)
        output_path = output_dir / f"{table_name}.parquet"
    
        # Check for and verify an existing output file and skip if exists
        # with correct record count.
        if output_path.exists():
            print(f"  Parquet file already exists: {output_path}")
            try:
                df = pd.read_parquet(output_path)
                parquet_record_count = len(df)
                print(f"  Existing parquet file has {parquet_record_count} records.")
                if parquet_record_count == record_count:
                    print("  Skipping this day - parquet file with correct record count already exists.")
                    continue
                else:
                    print(f"  Record count mismatch: {parquet_record_count} != {record_count}")
                    print("  File will be recreated.")
            except Exception as e:
                # Probably an invalid parquet file
                print(e)
        
        # Get data for the day from the TAP service.
        sql = f"SELECT * FROM ppdb.{table_name} WHERE FLOOR({ts_col}) = {day}"

        # Run the SQL job.
        job_start = time.time()
        job = AsyncTAPJob.create(
            service.baseurl,
            "SELECT * FROM ppdb.DiaObject WHERE FLOOR(validityStartMjdTai) >= 61090.0 AND FLOOR(validityStartMjdTai) < 61091.0",
            RESPONSEFORMAT="application/vnd.apache.parquet",
            session=session
        )
        job = job.run().wait()
        job_end = time.time()
        job_elapsed = job_end - job_start
        print(f"  Job took {job_elapsed:.2f} seconds")

        # Fetch the data.
        fetch_start = time.time()
        response = session.get(job.result_uri, stream=True)
        table_data = Table.read(io.BytesIO(response.content), format="parquet.votable")
        fetch_end = time.time() 
        fetch_elapsed = fetch_end - fetch_start
        print(f"  Fetch took {fetch_elapsed:.2f} seconds")
            
        # Write the entire day's data to a parquet file.
        parq_start = time.time()
        table_data.write(output_path, format="parquet", overwrite=True)
        parq_end = time.time()
        parq_elapsed = parq_end - parq_start
        print(f"  Wrote table data to '{output_path}' in {parq_elapsed:.2f} seconds\n")

    export_end = time.time()
    export_elapsed = export_end - export_start
    print(f"Export of {table_name} completed in {export_elapsed:.0f} seconds.")

In [ ]:
export_ppdb_table("DiaObject")

In [ ]:
export_ppdb_table("DiaSource")

<h2>Tests</h2>

In [ ]:
# runner = VOTableJobRunner(service)
# runner.execute("SELECT COUNT(*) FROM ppdb.DiaObject")

In [ ]:
# runner = VOParquetJobRunner(service)
# runner.execute("SELECT COUNT(*) FROM ppdb.DiaObject")

Test timing of VOTable vs VOParquet result

In [ ]:
sql = "SELECT * FROM ppdb.DiaObject WHERE FLOOR(validityStartMjdTai) >= 61090.0 AND FLOOR(validityStartMjdTai) < 61091.0"

# VOTable
print("Testing VOTable job...")
runner = VOTableJobRunner(service)
runner.execute(sql)

# VOParquet
print("Testing VOParquet job...")
runner = VOParquetJobRunner(service)
runner.execute(sql)